# The RAG Workshop

This is the workshop notebook for **GDG on Campus, Manipal University Jaipur (GDG MUJ)**.

We will build a **multi-PDF Retrieval-Augmented Generation (RAG) chatbot**. Upload your own PDF files (one or many), retrieve the most relevant passages across the uploaded documents, and give those passages to an LLM as evidence. **No sample PDFs are generated or required.**

**The key experiment:** ask the same document-specific question twice — first without the PDFs (LLM only), then with retrieved passages (RAG). We will also inspect *which PDF and page* supplied the answer.

> Run the notebook from top to bottom in Google Colab. Both text-based and **scanned / image-only PDFs** work: pages with no selectable text are read automatically with OCR (Tesseract).

## Part 01 · Setup & PDF Extraction

**Goal:** install the tools, connect to Groq, and extract text from *every PDF* we upload.

### Step 1 · Install dependencies

In [1]:

!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1
!pip install -q pymupdf sentence-transformers groq pytesseract pillow
!tesseract --version | head -1

### Step 2 · Import libraries

In [3]:
import io
import numpy as np
import pymupdf
import pytesseract
from PIL import Image
from sentence_transformers import SentenceTransformer
from groq import Groq
from google.colab import files, userdata

### Step 3 · Connect to the LLM

Add a Colab Secret named `GROQ_API_KEY` (key icon in the Colab left sidebar). The key stays out of the notebook code.

In [4]:
try:
    key = userdata.get("GROQ_API_KEY")
except Exception:
    key = None

if not key:
    raise ValueError(
        "GROQ_API_KEY not found. Add it in Colab Secrets (key icon in the left sidebar) "
        "and turn ON 'Notebook access' for it, then run this cell again."
    )

client = Groq(api_key=key)
MODEL = "openai/gpt-oss-20b"

def llm(prompt, system="You are a helpful assistant."):
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt}
            ],
            max_completion_tokens=2048,
            reasoning_effort="low"
        )
    except Exception as error:
        return f"[LLM error: {error}]"

    answer = (response.choices[0].message.content or "").strip()
    return answer or "[The model returned an empty answer. Try again or rephrase the question.]"

print(llm("Reply with: LLM connection successful."))

### Step 4 · Upload your own PDFs

This is the only document input step: choose PDF files from your computer in the upload window. You can select multiple files together. The notebook does **not** create or use pre-filled sample PDFs.

For the clearest multi-document demonstration, select at least two PDFs. One PDF also works, but cross-document retrieval can only be demonstrated when multiple files are uploaded. Scanned PDFs are fine — they are read with OCR in the next step.

In [7]:

uploaded = files.upload()


pdf_files = [name for name in uploaded if name.lower().endswith(".pdf")]
other_files = [name for name in uploaded if not name.lower().endswith(".pdf")]

if not pdf_files:
    raise ValueError("No PDF files were uploaded. Run this cell again and select one or more PDFs.")

print(f"\nUploaded {len(pdf_files)} PDF(s):")
for name in pdf_files:
    print("✓", name)

if other_files:
    print("\nIgnored non-PDF files:")
    for name in other_files:
        print("-", name)

if len(pdf_files) == 1:
    print("\nTip: upload 2 or more PDFs to demonstrate retrieval across different documents.")

### Step 5 · Extract text (with OCR fallback) and keep source information

For every page we first try to read the PDF's own text layer. If a page has (almost) no selectable text — a scan, a photo, or a slide exported as an image — we render it as an image and read it with **OCR**.

We store each page with its **PDF filename and page number**. This metadata will travel with every chunk, so RAG can show exactly where a retrieved passage came from.

In [8]:
OCR_LANG = "eng"
OCR_DPI = 200
MIN_TEXT_CHARS = 30

def ocr_page(page):
    pix = page.get_pixmap(dpi=OCR_DPI, alpha=False)
    image = Image.open(io.BytesIO(pix.tobytes("png")))
    return pytesseract.image_to_string(image, lang=OCR_LANG).strip()

pages = []
ocr_pages = 0
problem_files = []
ocr_broken = False

for filename in pdf_files:
    before = len(pages)
    try:
        with pymupdf.open(filename) as pdf:
            if pdf.needs_pass:
                print(f"Skipped {filename}: password-protected PDF.")
                problem_files.append(filename)
                continue

            for page_number, page in enumerate(pdf, start=1):
                page_text = page.get_text("text").strip()
                used_ocr = False

                if len(page_text) < MIN_TEXT_CHARS and not ocr_broken:
                    try:
                        ocr_text = ocr_page(page)
                    except Exception as error:
                        ocr_broken = True
                        print(f"OCR unavailable ({error}). Re-run the install cell (Step 1).")
                    else:
                        if len(ocr_text) > len(page_text):
                            page_text, used_ocr = ocr_text, True

                if page_text:
                    pages.append({
                        "source": filename,
                        "page": page_number,
                        "text": page_text,
                        "ocr": used_ocr
                    })
                    ocr_pages += used_ocr
    except Exception as error:
        print(f"Could not read {filename}: {error}")

    if len(pages) == before and filename not in problem_files:
        problem_files.append(filename)

if not pages:
    raise ValueError("No text could be extracted from any PDF (even with OCR). Check the files.")

print(f"\nText extracted from {len(pages)} pages across {len(pdf_files)} PDF(s) "
      f"({ocr_pages} page(s) read with OCR).")
for filename in pdf_files:
    mine = [p for p in pages if p["source"] == filename]
    print(f"{filename}: {len(mine)} pages ({sum(p['ocr'] for p in mine)} via OCR)")

if problem_files:
    print("\nNo text found in:", problem_files)

## Part 02 · Chunking and Embeddings

**Goal:** split every page into small overlapping passages, then convert each passage into a vector so that meaning can be compared mathematically.

### Step 6 · Write the chunk function

We split by words (not arbitrary character positions), and keep a small overlap so a fact near a chunk boundary is less likely to be lost.

In [10]:
def split_into_chunks(text, chunk_size=140, overlap=25):
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Use chunk_size > 0 and 0 <= overlap < chunk_size.")

    words = text.split()
    step = chunk_size - overlap
    chunks_out = []
    for start in range(0, len(words), step):
        chunks_out.append(" ".join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words):
            break
    return chunks_out

### Step 7 · Chunk all pages (from all PDFs)

Each chunk keeps its original filename and page number. The index is shared across the entire upload — not one index per PDF.

We also build a **context header** (`"Course Handout 2026 · page 1"`) and put it in front of the text that gets embedded. Without it, a chunk has no idea which document it came from, so a question like *"what course is this handout for?"* matches the wrong pages.

In [11]:
import re

def clean_name(filename):
    name = re.sub(r"\.pdf$", "", filename, flags=re.I)
    name = re.sub(r"\s*\(\d+\)$", "", name)
    return name.replace("_", " ").strip()

chunks = []

for page in pages:
    page_chunks = split_into_chunks(page["text"])
    for chunk_number, passage in enumerate(page_chunks, start=1):
        chunks.append({
            "text": passage,
            "embed_text": f"{clean_name(page['source'])} · page {page['page']}\n{passage}",
            "source": page["source"],
            "page": page["page"],
            "chunk": chunk_number,
            "ocr": page["ocr"]
        })

print("Total chunks:", len(chunks))
print("\nExample chunk:")
print(chunks[0]["embed_text"][:600])
print("\nSource:", chunks[0]["source"], "| Page:", chunks[0]["page"])

### Step 8 · Load the embedding model

In [12]:
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
get_dim = getattr(embed_model, "get_embedding_dimension", None) or embed_model.get_sentence_embedding_dimension
print("Embedding dimensions:", get_dim())

### Step 9 · Embed every chunk

An embedding is a numerical representation of meaning. We normalize the vectors so their dot product is cosine similarity.

In [13]:
chunk_texts = [item["embed_text"] for item in chunks]

embeddings = embed_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding matrix:", embeddings.shape)
print("One vector per chunk:", len(embeddings) == len(chunks))

## Part 03 · Similarity and Retrieval

**Goal:** turn a question into an embedding, compare it with every PDF chunk, and retrieve the closest passages — including passages from different PDFs.

### Step 10 · Similarity guessing game

Related sentences should have a higher cosine similarity than unrelated sentences. Scores are similarities, **not percentages or probabilities**.

In [14]:
sentences = [
    "The cat sat on the sofa.",
    "A kitten is resting on the couch.",
    "The stock market crashed today."
]

vectors = embed_model.encode(sentences, normalize_embeddings=True)
print("cat vs kitten:", round(float(vectors[0] @ vectors[1]), 3))
print("cat vs stocks:", round(float(vectors[0] @ vectors[2]), 3))

### Step 11 · Write the search function

Pure embedding search fails on **broad or vague questions** (e.g. *"what coding topics are contained?"*) and on **OCR'd code/lab-record pages**: every score comes out low (~0.15), so nothing looks relevant.

So we use **hybrid search**: meaning-based (embeddings) **+** keyword-based (BM25), merged with Reciprocal Rank Fusion. We can also search with several rewritten versions of the question and merge the results.

In [15]:
import re, math
from collections import Counter

STOP = set("""a an the of in on to for and or is are was were what which who whom how all do does did
this that these those with from by be as at it its contained contain about tell me give list show
please can you i my our pdf pdfs document documents file files""".split())

def tokenize(text):
    return [w for w in re.findall(r"[a-z0-9_]+", text.lower()) if w not in STOP and len(w) > 1]


bm_docs = [tokenize(c["embed_text"]) for c in chunks]
N_DOCS  = len(bm_docs)
AVG_LEN = sum(len(d) for d in bm_docs) / max(N_DOCS, 1)
DF  = Counter(w for d in bm_docs for w in set(d))
TFS = [Counter(d) for d in bm_docs]

def bm25_scores(query, k1=1.5, b=0.75):
    scores = np.zeros(N_DOCS)
    for w in set(tokenize(query)):
        if w not in DF:
            continue
        idf = math.log(1 + (N_DOCS - DF[w] + 0.5) / (DF[w] + 0.5))
        for i, tf in enumerate(TFS):
            f = tf.get(w, 0)
            if f:
                scores[i] += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * len(bm_docs[i]) / AVG_LEN))
    return scores

def search(question, k=5, extra_queries=None):
    """Hybrid retrieval. Returns top-k chunks; hit['score'] = best cosine similarity."""
    if not question.strip():
        raise ValueError("Enter a question.")

    queries = [question] + [q for q in (extra_queries or []) if q.strip()]
    fused   = np.zeros(N_DOCS)
    best_cos = np.full(N_DOCS, -1.0)
    best_kw  = np.zeros(N_DOCS)

    for q in queries:
        qv  = embed_model.encode([q], normalize_embeddings=True)[0]
        cos = embeddings @ qv
        kw  = bm25_scores(q)
        best_cos = np.maximum(best_cos, cos)
        best_kw  = np.maximum(best_kw, kw)

        for ranking in (np.argsort(cos)[::-1], np.argsort(kw)[::-1] if kw.max() > 0 else []):
            for rank, idx in enumerate(ranking[:50], start=1):
                fused[idx] += 1.0 / (60 + rank)

    results = []
    for index in np.argsort(fused)[::-1][:k]:
        hit = chunks[index].copy()
        hit["score"]   = float(best_cos[index])
        hit["keyword"] = float(best_kw[index])
        hit["fused"]   = float(fused[index])
        results.append(hit)
    return results

### Step 12 · Inspect retrieval and set the threshold

A similarity score is **not** an answer-confidence score. With `all-MiniLM-L6-v2` on real documents (especially OCR'd code), a correct passage can score only **0.10–0.30**.

A fixed cutoff like `0.25` silently throws away *everything* for vague questions. So we use a **very low floor** (`0.08`) and keep a passage if it passes the floor **or** it shares keywords with the question. The LLM then decides if the evidence really answers it.

In [16]:
MIN_SCORE = 0.08
TOP_K = 8

def is_relevant(hit):
    return hit["score"] >= MIN_SCORE or hit["keyword"] > 0

def ask_input(prompt):
    """input() that never crashes the cell on empty / interrupted input."""
    try:
        return input(prompt).strip()
    except (EOFError, KeyboardInterrupt):
        return ""

probe_question = ask_input("Ask a question whose answer is somewhere in your PDFs: ")
if not probe_question:
    raise ValueError("Enter a question, then run this cell again.")

for rank, hit in enumerate(search(probe_question, k=TOP_K), start=1):
    status = "KEEP" if is_relevant(hit) else "FILTER"
    print(f"\n#{rank} | cosine={hit['score']:.3f} | keyword={hit['keyword']:.2f} | {status}")
    print(f"PDF: {hit['source']} | Page: {hit['page']}")
    print(hit["text"][:350], "...")

### Step 13 · What retrieval is doing

1. The question becomes an embedding.
2. The embedding is compared with every stored chunk.
3. The highest-scoring chunks are ranked.
4. The threshold filters out weak matches.
5. The surviving chunks become the evidence passed to the LLM.

**Important:** a high retrieval score does not prove that a passage answers the question. Always inspect the retrieved text and source pages.

## Part 04 · Generation & Multi-PDF RAG Chatbot

**Goal:** compare an LLM without document access against the same LLM with retrieved evidence, then chat across the uploaded PDFs.

### Step 14 · Without RAG: ask the LLM directly

The model receives only the question — **none of the uploaded PDF content**. It may know general facts, but it cannot verify what your PDFs specifically say.

In [17]:
def answer_without_rag(question):
    prompt = f"""You do not have access to the user's uploaded PDFs.
Answer the question using general knowledge only.
If the question asks what a specific uploaded document says, clearly state that
those documents were not provided to you and you cannot verify their contents.
Do not pretend you have read the PDFs.

Question: {question}
"""
    return llm(prompt)

### Step 15 · Build the grounded prompt

The prompt includes the retrieved passages and labels each one with a source ID. The model must answer from those passages and cite the source IDs it used.

In [18]:
def build_prompt(question, retrieved, broad=False):
    context = "\n\n".join(
        f"[S{i}] PDF: {hit['source']} | Page: {hit['page']}\n{hit['text']}"
        for i, hit in enumerate(retrieved, start=1)
    )
    style = ("This is a broad question: give a well-organised list/summary covering ALL the excerpts, "
             "grouped by document.\n") if broad else ""

    return f"""Answer the question using ONLY the source excerpts below.
The PDF filename and page number of each excerpt are part of the evidence; you may use them
(for example, to say which document something comes from).
The text may come from OCR of scanned pages, so ignore small OCR typos.
{style}If the excerpts only partly answer the question, give what they do say and state what is missing.
Only if they contain nothing relevant, say: "I couldn't find this in the uploaded PDFs."
Do not use outside knowledge or make up details.
Cite factual claims using the source labels, like [S1] or [S2].
Only cite source labels that appear below.

SOURCE EXCERPTS:
{context}

QUESTION: {question}

ANSWER:
"""

### Step 16 · Write the RAG function

RAG = **retrieve first, generate second**. Two upgrades make it handle real-world questions:

1. **Query expansion** – the LLM rewrites a vague question into a few document-style search queries.
2. **Broad-question mode** – for questions like *"what topics are covered / summarise / list all"*, similarity search can't work (the answer is spread over every page), so we pass a short excerpt from **each page** instead.

In [19]:
BROAD_PATTERN = re.compile(
    r"\b(topics?|summar\w*|overview|list|what all|everything|all the|each|contents?|"
    r"covered|programs?|experiments?|chapters?|sections?|main points|about these)\b", re.I)

def expand_query(question, n=3):
    """Ask the LLM for short alternative search queries. Falls back to [] on any problem."""
    reply = llm(
        f"Rewrite this question as {n} short, different search queries that would match text "
        f"inside the documents (use likely keywords, technical terms and synonyms). "
        f"Return one query per line, no numbering, no extra text.\n\nQuestion: {question}",
        system="You write search queries.")
    if reply.startswith("[") :
        return []
    lines = [l.strip(" -•*\t") for l in reply.splitlines() if l.strip()]
    return lines[:n]

def coverage_context(question, per_page_chars=500, max_pages=30):
    """One excerpt per page (all documents represented), most relevant pages first."""
    hits = search(question, k=len(chunks))
    seen, picked = set(), []
    for h in hits:
        key = (h["source"], h["page"])
        if key not in seen:
            seen.add(key)
            h = h.copy(); h["text"] = h["text"][:per_page_chars]
            picked.append(h)
    return picked[:max_pages]

def ask(question, k=TOP_K, show_sources=True):
    broad = bool(BROAD_PATTERN.search(question))
    extra = expand_query(question)

    if broad:
        retrieved = coverage_context(question)
    else:
        hits = search(question, k=k, extra_queries=extra)
        retrieved = [h for h in hits if is_relevant(h)] or hits[:3]

    if show_sources:
        print("MODE:", "broad (one excerpt per page)" if broad else "targeted (hybrid search)")
        if extra:
            print("EXPANDED QUERIES:", " | ".join(extra))
        print("RETRIEVED EVIDENCE")
        for i, hit in enumerate(retrieved, start=1):
            ocr_note = " (OCR)" if hit.get("ocr") else ""
            print(f"[S{i}] {hit['source']} | page {hit['page']}{ocr_note} "
                  f"| cosine {hit['score']:.3f} | keyword {hit['keyword']:.2f}")
        print()

    return llm(build_prompt(question, retrieved, broad=broad))

### Step 17 · Compare: LLM-only vs RAG

For the clearest demo, ask about a **specific detail** in one of the PDFs — a named method, date, statistic, policy, or example. A generic question may be answerable from the LLM's general knowledge, which is not a fair test of document retrieval.

In [20]:
test_question = ask_input("\nEnter one document-specific question: ")
if not test_question:
    raise ValueError("Enter a question, then run this cell again.")

print("=" * 75)
print("WITHOUT RAG — the LLM has not received the PDFs")
print("=" * 75)
print(answer_without_rag(test_question))

print("\n" + "=" * 75)
print("WITH MULTI-PDF RAG — retrieved passages are supplied as evidence")
print("=" * 75)
print(ask(test_question, show_sources=True))

### Step 18 · Chat with all your PDFs

Ask follow-up questions across the complete uploaded collection. Enter `exit` to stop.

In [21]:
print("Chat started. Type your question and press Enter. Type 'exit' to stop.")
while True:
    question = ask_input("\nYou: ")

    if question.lower() in ("exit", "quit"):
        print("Goodbye!")
        break

    if not question:
        print("(empty input - type a question or 'exit')")
        continue

    print("\nBot:", ask(question, show_sources=True))

### Step 21 · Verify GitHub Connection & Authentication

In [25]:
import subprocess

print("=== Git Local Configuration ===")
!git config --list --global

print("\n=== Checking Connection & Authentication to GitHub ===")
try:
    # Try to connect via SSH first to see if keys are configured
    ssh_check = subprocess.run(
        ["ssh", "-T", "git@github.com"],
        capture_output=True,
        text=True,
        timeout=5
    )
    print(ssh_check.stderr.strip())
except Exception as e:
    print(f"SSH check skipped or timed out: {e}")

# Validate variables configured in Step 20
try:
    from google.colab import userdata
    # Check if a token is saved in Colab secrets
    has_secret = "GITHUB_TOKEN" in dir(userdata) or any(k == "GITHUB_TOKEN" for k in os.environ)
    print(f"\nColab Secrets check for GITHUB_TOKEN: {'Found' if has_secret else 'Not Found (Using local notebook variable)'}")
except Exception:
    pass

print(f"Configured Username: {GITHUB_USERNAME}")
print(f"Target Repository: {GITHUB_REPO}")


### Step 22 · Export and Push the Colab Notebook to GitHub

In [ ]:
import json
import os
import shutil

# Securely define the token in memory ONLY (This will not be saved inside the exported .ipynb)
MY_GITHUB_TOKEN = "" #@param {type:"string"}
notebook_filename = "RAG_Workshop_Notebook.ipynb"

# Step 1: Securely save progress and scrub ALL traces of the token from the .ipynb file
try:
    from google.colab import _message
    nb_json = _message.blocking_request('get_ipynb', request='', timeout_sec=10)
    nb_data = nb_json['ipynb']

    # Traverse cells in JSON to scrub the raw token completely from both code lines and outputs
    for cell in nb_data.get('cells', []):
        if cell.get('cell_type') == 'code':
            # Clean cell source code
            source_lines = cell.get('source', [])
            cleaned_source = []
            for line in source_lines:
                # If the line defines the token, scrub it
MY_GITHUB_TOKEN = "" #@param {type:"string"}
MY_GITHUB_TOKEN = "" #@param {type:"string"}
                else:
                    cleaned_source.append(line)
            cell['source'] = cleaned_source

            # Clean cell outputs to ensure the token is never printed/cached in metadata
            if 'outputs' in cell:
                cell['outputs'] = []

    with open(notebook_filename, 'w') as f:
        json.dump(nb_data, f, indent=2)
    print(f"[Step 1] ✓ Successfully saved notebook and scrubbed all traces of '{SECRET_TOKEN[:5]}...' from the file.")
except Exception as e:
    print(f"[Step 1] Error saving notebook: {e}")

# Step 2: Use the token from memory
token = SECRET_TOKEN.strip()

if not token:
    print("\n[Step 2] ✗ Personal Access Token (PAT) not found in memory. Please provide it above.")
else:
    print("\n[Step 3] Purging local git repository to clean commit history...")

    # Deleting local .git directory to purge compromised history
    if os.path.exists(".git"):
        shutil.rmtree(".git")

    # Re-initialize local repository
    !git init -b main
    !git config --global user.name "titasgarai"
    !git config --global user.email "titasgarai@users.noreply.github.com"

    repo_url = "github.com/titasgarai/GDGMUJ_RAG_Workshop.git"
    authenticated_url = f"https://{token}@{repo_url}"

    !git remote add origin {authenticated_url}

    # Stage only clean files
    !git add README.md {notebook_filename} 01_RAG_Workshop_Event_Details.pdf 02_RAG_Workshop_Guidelines_and_Rules.pdf 03_GDG_MUJ_Past_Events.pdf
    !git commit -m "Clean initial commit: Multi-PDF RAG workshop materials"

    print("\n[Step 4] Pushing clean initial history to GitHub...")
    !git push -u origin main --force